# Pipeline Experimental para Segmentação Médica Tridimensional com MONAI e PyTorch

### Segmentação Hepática 3D · Medical Segmentation Decathlon — Task03_Liver

**Versão 1.0 — Baseline (U-Net 3D)**

**Ambiente oficial:** Google Colab · Python 3.12 · PyTorch 2.x · MONAI 1.6

---

> ⚠️ **Aviso**
>
> Este notebook implementa a visão científica, arquitetural e metodológica descrita no
> *Documento de Projeto Científico (DPC)*. O pipeline destina-se **exclusivamente a
> pesquisa, ensino e desenvolvimento tecnológico**. Embora utilize dados médicos reais e
> metodologias consolidadas na literatura, **não constitui um dispositivo médico** nem deve
> ser usado para decisão clínica sem validação específica, aprovação ética quando aplicável
> e conformidade regulatória vigente.

---

#### Sobre este notebook

Este é um **notebook monolítico organizado em camadas**: todo o fluxo experimental — do
ambiente à análise dos resultados — vive em um único arquivo, documentado em ordem
cronológica. A escolha é deliberada (DPC §5.2): facilita a **revisão metodológica** e a
compreensão por pesquisadores com pouca experiência em engenharia de software, sem abrir mão
da **modularidade lógica** entre as seções.

O texto foi escrito para ser lido tanto por profissionais de tecnologia quanto por
profissionais de saúde. Sempre que uma decisão metodológica for tomada, ela será
**justificada explicitamente** — nada relevante fica implícito no código (princípio da
*Transparência*, DPC §5.1).

## 🗺️ Roadmap do notebook

O pipeline é uma sequência de camadas independentes (DPC §5.5), cada uma recebendo uma
entrada e produzindo a saída da etapa seguinte, **sem dependências circulares**. No notebook,
essas camadas aparecem como seções sequenciais:

| # | Seção | Status |
|---|-------|--------|
| **1** | **Configuração do Ambiente** — dependências, GPU/CPU, proveniência | ✅ **implementada** |
| **2** | **Configuração Global** — sementes, determinismo e hiperparâmetros | ✅ **implementada** |
| **3** | **Preparação e Exploração do Dataset** | ✅ **implementada** |
| 4 | Pré-processamento (transforms MONAI) | ⏳ pendente |
| 5 | DataLoaders | ⏳ pendente |
| 6 | Definição da Arquitetura (U-Net 3D) | ⏳ pendente |
| 7 | Treinamento | ⏳ pendente |
| 8 | Inferência (*sliding window*) | ⏳ pendente |
| 9 | Avaliação Quantitativa e Qualitativa | ⏳ pendente |
| 10 | Visualização e Persistência | ⏳ pendente |
| 11 | Conclusões | ⏳ pendente |

> 💡 Esta entrega contém as **Seções 1 a 3**. As demais seções serão adicionadas
> incrementalmente, cada uma revisada antes de prosseguir.

# 1 · Configuração do Ambiente

A **primeira camada** do pipeline (DPC §5.5) tem uma responsabilidade simples de enunciar,
porém decisiva para a ciência do projeto: **deixar o ambiente de execução pronto e
documentado**. Nada de dados ou modelos ainda — apenas a fundação sobre a qual todo o resto
será construído.

**Por que isso vem antes de tudo?** Porque a reprodutibilidade (DPC §6.6) começa aqui. Um
resultado científico só tem valor se puder ser **reconstruído de forma independente** — e
isso exige saber, com precisão, *em que ambiente* ele foi produzido: qual versão de cada
biblioteca, qual dispositivo de hardware, em que data.

Nesta seção nós vamos:

1. **Detectar o ambiente** de execução (Google Colab ou máquina local);
2. **Instalar as dependências** com versões fixadas, reutilizando o PyTorch já presente no Colab;
3. **Configurar o dispositivo** de computação (GPU se disponível, com fallback para CPU);
4. **Registrar a proveniência** — um "cartão de identidade" do ambiente, para logging dos experimentos.

> 📌 **Fora do escopo desta seção.** As **sementes aleatórias** (`set_determinism`) e os
> **hiperparâmetros** pertencem conceitualmente à camada de *Configuração Global* e serão
> definidos na **Seção 2**. Mantemos aqui apenas o que diz respeito ao *ambiente*, para que
> cada seção tenha uma responsabilidade única e clara (DPC §5.1, Modularidade).

## 1.1 · Detecção do ambiente

O **ambiente oficial do projeto é o Google Colab** (DPC §5.3), que reduz barreiras de
infraestrutura e dá acesso relativamente simples a GPUs. Ainda assim, escrevemos o notebook
para funcionar também localmente. A célula abaixo apenas descobre **onde** estamos rodando —
essa informação orienta a instalação de dependências no passo seguinte.

In [ ]:
# Detecta se o notebook está sendo executado no Google Colab.
# A biblioteca `google.colab` só existe dentro do Colab; sua ausência indica ambiente local.
try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

print(f"Executando no Google Colab? {'sim' if EM_COLAB else 'não (ambiente local)'}")

## 1.2 · Instalação de dependências

Seguimos o princípio do DPC §5.4: **em vez de reimplementar componentes fundamentais, usamos
o MONAI**, que já oferece transformações médicas, arquiteturas 3D, inferência por janelas
deslizantes, métricas e funções de perda — tudo validado pela comunidade.

**Estratégia de versões (DPC §6.6 — "fixar versões sempre que possível"):**

- **PyTorch:** *não* é reinstalado. O Colab já traz uma build de PyTorch casada com a versão
  de CUDA do ambiente; reinstalá-lo é a principal fonte de incompatibilidades. Reutilizamos o
  que está presente e apenas **verificamos** a versão no passo 1.4.
- **MONAI:** fixado em `monai==1.6.0` — **exatamente a versão que o Colab já entrega
  pré-instalada** no momento desta escrita. Por que fixar em algo já presente? Porque o pin
  (a) *documenta* qual versão foi testada e (b) protege o experimento caso, no futuro, o Colab
  passe a trazer uma versão diferente. Como a versão fixada coincide com a pré-instalada, o
  comando é **idempotente** (o pip responde "already satisfied" e não altera nada) — evitando
  downgrades, reinstalações e a necessidade de reiniciar a sessão.
- **nibabel:** biblioteca de leitura/escrita de arquivos **NIfTI** (`.nii.gz`), o formato dos
  volumes do dataset. O MONAI a utiliza internamente para carregar as imagens.

> 🔭 **Compatibilidade futura (DPC §9.2).** A v1.0 mantém o conjunto de dependências enxuto.
> Quando o projeto evoluir para **Transformers** (v2.0: UNETR, SwinUNETR), bastará acrescentar
> extras como `einops` a esta mesma célula — a estrutura da seção não muda. (No Colab atual, o
> `einops` já vem instalado, como o inventário do passo 1.4 confirma.)

In [ ]:
# Versão fixada para reprodutibilidade (DPC §6.6). Coincide com a que o Colab entrega
# pré-instalada, então a instalação é idempotente: sem downgrade, sem reiniciar a sessão.
MONAI_VERSION = "1.6.0"

if EM_COLAB:
    import subprocess, sys
    pacotes = [f"monai=={MONAI_VERSION}", "nibabel"]
    print("Garantindo dependências:", ", ".join(pacotes))
    # O PyTorch do Colab é preservado (não aparece na lista, portanto não é reinstalado).
    resultado = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *pacotes],
        capture_output=True, text=True,
    )
    if resultado.returncode == 0:
        print("Dependências prontas.")
    else:
        # Só mostramos os logs do pip se algo falhar — mantém a saída limpa no caso comum.
        print("Falha na instalação. Logs do pip:")
        print(resultado.stdout)
        print(resultado.stderr)
else:
    # Em ambiente local, presume-se que as dependências já estejam instaladas
    # (ex.: via `pip install monai==1.6.0 nibabel`). Apenas avisamos.
    print("Ambiente local: verifique manualmente que 'monai' e 'nibabel' estão instalados.")

## 1.3 · Configuração do dispositivo (GPU / CPU)

Modelos 3D exploram simultaneamente altura, largura e profundidade dos exames, preservando as
relações anatômicas — ao custo de **muito mais memória e computação** (DPC §3.4). Na prática,
isso torna a **GPU essencial** para treinar em tempo razoável.

O Colab oferece GPU, mas sua **disponibilidade é variável** (DPC §10.1). Por isso a célula
abaixo:

- seleciona a **GPU** (`cuda`) quando disponível e imprime seu nome, memória e a versão de CUDA;
- faz um **fallback gracioso para CPU**, com um **aviso explícito** de que o treinamento será
  lento — evitando que alguém treine na CPU sem perceber.

A variável `device` resultante é **reutilizada por todas as seções seguintes** (modelo, dados
e inferência serão enviados para ela).

> 💡 **Como habilitar a GPU no Colab:** menu *Ambiente de execução → Alterar o tipo de
> ambiente de execução → Acelerador de hardware → GPU*.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    nome_gpu = torch.cuda.get_device_name(0)
    memoria_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU disponível: {nome_gpu}")
    print(f"   Memória total : {memoria_gb:.1f} GB")
    print(f"   Versão de CUDA: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print("⚠️  Nenhuma GPU detectada — usando CPU.")
    print("    O treinamento de um modelo 3D na CPU é MUITO lento e não é recomendado.")
    print("    No Colab: Ambiente de execução → Alterar o tipo de ambiente → GPU.")

print(f"\nDispositivo selecionado: {device}")

## 1.4 · Verificação e registro de proveniência

Reprodutibilidade não é só *fixar* versões — é **registrá-las**. Cada experimento do projeto
deve poder ser reconstruído a partir de um "cartão de identidade" do ambiente: versões de
bibliotecas, dispositivo e data de execução (DPC §6.6 e §7.2).

Fazemos isso em dois passos:

1. **Inventário completo via MONAI.** Usamos o componente **oficial** `monai.config.print_config()`
   — que já lista as versões do MONAI, do PyTorch e de todas as dependências opcionais. Seguindo
   o princípio de *não reimplementar o que o MONAI já resolve* (DPC §5.4), preferimos essa função
   a montar uma verificação manual.
2. **Dicionário `AMBIENTE`.** Consolidamos os campos essenciais em uma estrutura que as seções
   futuras (treinamento, avaliação) anexarão ao registro de cada experimento.

In [ ]:
import monai

# Inventário oficial do MONAI: versões do MONAI, PyTorch, NumPy e dependências opcionais.
# (Com a instalação idempotente do passo 1.2, este import é direto — sem reiniciar a sessão.)
monai.config.print_config()

In [ ]:
import platform
from datetime import datetime, timezone

# "Cartão de identidade" do ambiente — reutilizado no logging de cada experimento (DPC §6.6/§7.2).
AMBIENTE = {
    "data_execucao_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda if torch.cuda.is_available() else None,
    "monai": monai.__version__,
    "dispositivo": str(device),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "em_colab": EM_COLAB,
    "versao_notebook": "1.0-baseline",
}

print("Registro de proveniência do ambiente")
print("-" * 44)
for chave, valor in AMBIENTE.items():
    print(f"{chave:>18} : {valor}")

## ✅ Resumo da Seção 1

O ambiente está pronto e **documentado**:

- detectamos o contexto de execução (`EM_COLAB`);
- fixamos o **MONAI 1.6.0** e garantimos o **nibabel**, reutilizando o **PyTorch do Colab**;
- selecionamos o **dispositivo** de computação (`device`), com fallback e aviso para CPU;
- registramos a **proveniência** do ambiente no dicionário `AMBIENTE`.

Essas variáveis — `device` e `AMBIENTE` — atravessam todo o notebook e são a base da
reprodutibilidade dos experimentos.

**➡️ Próxima seção — Configuração Global.** Definiremos a **semente aleatória** e o **modo
determinístico** (Python, NumPy, PyTorch e MONAI, via `set_determinism`), além dos
**hiperparâmetros globais** do pipeline. Só então começaremos a tocar nos dados.

> *"Tecnologias mudam. Frameworks evoluem. Modelos são substituídos. O rigor científico permanece."*
> — Manifesto Científico da Plataforma (DPC §11.4)

# 2 · Configuração Global

Se a Seção 1 preparou o *ambiente*, esta seção estabelece o **protocolo experimental** — o
conjunto de decisões que precisa permanecer **constante** para que os resultados sejam
comparáveis e reproduzíveis. O DPC é enfático (§5.1, §6): *"pequenas alterações no
pré-processamento, na divisão dos dados, nas métricas ou na configuração do treinamento podem
produzir diferenças substanciais nos resultados"*. Por isso essas decisões são declaradas
**uma única vez, em um só lugar**, antes de qualquer código de dados ou de modelo.

Esta seção tem duas responsabilidades:

1. **Controlar a aleatoriedade** — fixar as sementes e ativar o modo determinístico (§6.2);
2. **Centralizar os hiperparâmetros** — reunir num único objeto imutável os valores que
   definem o protocolo (§7.2, *"separação clara entre configuração e execução"*).

> 🧭 **Por que isso é o coração da comparabilidade (§9.2).** O projeto evolui em versões
> (1.0 U-Net → 1.1 CNNs modernas → 2.0 Transformers). Em todas elas, *a única variável
> deliberadamente modificada é a arquitetura*. Manter dados, pré-processamento, métricas e
> treinamento fixos aqui é o que garante que uma diferença de desempenho seja atribuível à
> **arquitetura**, e não a mudanças ocultas no protocolo.

## 2.1 · Controle de aleatoriedade

Redes neurais dependem de **acaso** em vários pontos: a inicialização dos pesos, a ordem de
embaralhamento dos dados, o *data augmentation*. Sem controlar essas fontes, duas execuções do
mesmo código produzem resultados diferentes — e um resultado que não se repete não pode ser
verificado por outro pesquisador.

A solução é **fixar uma semente (seed)**: um número que torna todas as sequências
"aleatórias" reprodutíveis. O DPC (§6.2) pede que a semente seja fixada para **Python, NumPy,
PyTorch e MONAI**, com modos determinísticos quando tecnicamente possível.

Em vez de semear cada biblioteca manualmente, usamos o componente **oficial**
`monai.utils.set_determinism()` (princípio §5.4 — não reimplementar). Com uma única chamada
ele semeia `random`, `numpy` e `torch`, e ativa o modo determinístico do cuDNN.

> ⚖️ **Uma nota honesta sobre limites (§6.2).** Mesmo com a semente fixa, pequenas diferenças
> numéricas podem surgir em GPU — algumas operações 3D não possuem implementação determinística.
> Por isso **não** forçamos `torch.use_deterministic_algorithms(True)`: isso interromperia o
> treino com erro em vez de apenas variar na última casa decimal. O DPC aceita explicitamente
> essa pequena variação de hardware como razoável.

In [ ]:
from monai.utils import set_determinism

# Semente mestra do projeto. Um valor único e documentado, reutilizado em todo o pipeline.
SEED = 0

# Semeia Python (random), NumPy e PyTorch de uma só vez e ativa o modo determinístico do cuDNN.
set_determinism(seed=SEED)

print(f"Determinismo ativado com SEED = {SEED}")
print("Sementes fixadas para: random (Python), NumPy e PyTorch; cuDNN em modo determinístico.")

## 2.2 · Hiperparâmetros globais

Aqui declaramos, num **único objeto**, todos os hiperparâmetros do protocolo experimental.
Reuni-los em um só lugar traz três benefícios diretos do DPC:

- **Comparabilidade (§5.1):** o protocolo fica visível e fixo; trocar a arquitetura nas
  versões futuras não exige tocar em nenhum destes valores.
- **Transparência (§5.1):** nada de "números mágicos" espalhados pelo código — toda constante
  tem nome e lugar.
- **Reprodutibilidade (§6.6):** este objeto é registrado junto ao experimento, permitindo
  reconstruí-lo com exatidão.

Usamos um `dataclass` **imutável** (`frozen=True`): uma vez definido, o `CONFIG` **não pode ser
alterado acidentalmente durante a execução** — uma salvaguarda alinhada ao princípio de que
cada experimento é uma unidade preservada, nunca sobrescrita (§8.3).

> 📖 **Modularidade (§5.1).** Declaramos os valores aqui, mas a *justificativa metodológica* de
> cada um pertence à sua seção: o significado de `pixdim`, `a_min/a_max` e `spatial_size` é
> explicado no **Pré-processamento (Seção 4)**; as frações `val_split`/`test_split` da divisão
> por paciente, na **Preparação do Dataset (Seção 3)**; e o cronograma de treino, na **Seção 7**.
>
> ⚠️ **Revisão crítica (§8.2).** A `learning_rate = 1e-5` e o `max_epochs = 600` são herdados
> do script de referência e mantidos para estabelecer a *baseline*. São candidatos naturais a
> ajuste futuro — mas qualquer mudança será uma **decisão registrada**, não silenciosa, para
> preservar a comparabilidade entre versões.

In [ ]:
from dataclasses import dataclass, asdict

# Protocolo experimental do projeto — constante entre versões (só a arquitetura muda).
@dataclass(frozen=True)
class Config:

    # — Reprodutibilidade —
    seed: int = SEED

    # — Pré-processamento (§6.3) · justificado na Seção 4 —
    pixdim: tuple[float, float, float] = (1.5, 1.5, 1.0)   # reamostragem espacial (mm)
    a_min: float = -200.0                                  # janela de intensidade (HU): mínimo
    a_max: float = 200.0                                   # janela de intensidade (HU): máximo
    b_min: float = 0.0                                     # intensidade normalizada: mínimo
    b_max: float = 1.0                                     # intensidade normalizada: máximo
    clip: bool = True                                      # recorta valores fora da janela
    spatial_size: tuple[int, int, int] = (128, 128, 64)    # tamanho do volume após redimensionar

    # — Dados e divisão (§6.1) · realizada na Seção 3 —
    val_split: float = 0.15     # fração do conjunto rotulado para validação (separação por paciente)
    test_split: float = 0.15    # fração do conjunto rotulado para teste (isolado até a avaliação final)
    batch_size: int = 1         # volumes 3D são grandes; lote pequeno cabe na memória da GPU
    num_workers: int = 2        # processos paralelos de carregamento de dados
    cache_rate: float = 1.0     # fração do dataset mantida em cache (MONAI CacheDataset)

    # — Treinamento (§6.4) · usado na Seção 7 —
    learning_rate: float = 1e-5   # taxa de aprendizado do otimizador
    weight_decay: float = 1e-5    # regularização L2
    max_epochs: int = 600         # número máximo de épocas (limites de tempo do Colab podem reduzir)
    val_interval: int = 1         # validar a cada N épocas

# Instância única e imutável usada por todo o pipeline.
CONFIG = Config()

print("Hiperparâmetros globais (CONFIG):")
print("-" * 44)
for chave, valor in asdict(CONFIG).items():
    print(f"{chave:>16} : {valor}")

## 2.3 · Registro do experimento

Reprodutibilidade se completa quando **ambiente** (Seção 1) e **protocolo** (esta seção) são
registrados juntos. Combinamos `AMBIENTE` e `CONFIG` num único dicionário — o "prontuário" do
experimento —, que as seções de treinamento e avaliação anexarão aos resultados salvos (§6.6,
§7.2, §8.3). Assim, qualquer execução pode ser reconstruída a partir do que ficou registrado.

In [ ]:
# Prontuário do experimento: ambiente (versões/dispositivo) + protocolo (hiperparâmetros).
REGISTRO_EXPERIMENTO = {
    "ambiente": AMBIENTE,          # definido na Seção 1
    "configuracao": asdict(CONFIG),
}

import json
print("Registro completo do experimento (a ser salvo junto aos resultados):")
print(json.dumps(REGISTRO_EXPERIMENTO, indent=2, ensure_ascii=False))

## ✅ Resumo da Seção 2

O protocolo experimental está estabelecido e registrado:

- fixamos a semente mestra (`SEED`) e ativamos o determinismo via `set_determinism` (§6.2);
- centralizamos os hiperparâmetros num `CONFIG` **imutável**, que fixa o protocolo entre
  versões e evita "números mágicos" (§5.1, §7.2);
- consolidamos ambiente + protocolo em `REGISTRO_EXPERIMENTO`, base do logging reproduzível (§6.6).

A partir daqui, todo o pipeline consome esses valores — **nenhuma seção seguinte redefine um
hiperparâmetro do protocolo**; elas apenas o utilizam.

**➡️ Próxima seção — Preparação e Exploração do Dataset.** Vamos localizar o MSD Task03_Liver,
**validar sua estrutura e integridade** (sem transformar nada ainda, §5.5) e realizar a divisão
por paciente em treino/validação/teste (§6.1), usando os `val_split`/`test_split` definidos aqui.

# 3 · Preparação e Exploração do Dataset

Esta é a camada de **Dados** (DPC §5.5). Sua responsabilidade é deliberadamente restrita:
**localizar o dataset, validar sua estrutura e integridade e organizá-lo em conjuntos** — mas
*sem transformar nada ainda*. Toda transformação (orientação, reamostragem, normalização) fica
para o Pré-processamento (Seção 4). Aqui apenas garantimos que os dados existem, estão íntegros
e corretamente pareados.

Trabalhamos com o **Medical Segmentation Decathlon — Task03_Liver** (DPC §4), um benchmark
público de CT hepática. Seguindo o princípio §6.1 — *"o dataset é mantido exatamente em sua
estrutura oficial; nenhum arquivo original é modificado"* —, esperamos a **estrutura oficial do
MSD**:

```
Task03_Liver/
├── imagesTr/     # volumes de CT rotulados   (liver_0.nii.gz, liver_1.nii.gz, ...)
├── labelsTr/     # máscaras correspondentes   (mesmos nomes de imagesTr)
├── imagesTs/     # volumes de teste SEM rótulo público
└── dataset.json  # metadados oficiais (modalidade, rótulos, contagens)
```

Nesta seção nós vamos:

1. **Montar o Google Drive** e localizar o dataset;
2. **Obter o dataset** — download *opcional e idempotente*, caso ainda não esteja no Drive;
3. **Validar a estrutura** oficial e ler os metadados (§7.1);
4. **Parear e conferir a integridade** de imagens e máscaras (§7.1);
5. **Dividir por paciente** em treino/validação/teste (§6.1);
6. **Explorar** visualmente um exemplo para confirmar que tudo faz sentido.

> ⚙️ **Uma decisão de nomenclatura.** O script de referência usava as chaves `vol`/`seg`.
> Adotamos aqui as chaves **`image`/`label`**, padrão nos exemplos oficiais do MONAI e no
> `DecathlonDataset` — o que reduz atrito ao reutilizar componentes do framework nas próximas
> seções.
>
> 🔒 **Sobre o `imagesTs`.** Como os rótulos oficiais do conjunto de teste do MSD **não são
> públicos**, não é possível calcular métricas sobre ele. Por isso (decisão registrada, §8.2)
> derivamos treino/validação/**teste** a partir do conjunto **rotulado** (`imagesTr`/`labelsTr`);
> o `imagesTs` é deixado intocado.

## 3.1 · Montagem do Google Drive e localização do dataset

O ambiente oficial é o Colab (§5.3), e a forma mais prática de disponibilizar dezenas de
gigabytes de imagens médicas é mantê-las no **Google Drive** e montá-lo no notebook. A célula
abaixo monta o Drive e define `DATA_DIR` — **o único caminho que você talvez precise ajustar**,
apontando para a raiz da pasta `Task03_Liver`.

In [ ]:
from pathlib import Path

# >>> AJUSTE AQUI, se necessário: caminho da raiz do dataset na estrutura oficial do MSD. <<<
DATA_DIR = "/content/drive/MyDrive/Task03_Liver"

if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_DIR = Path(DATA_DIR)
print("Diretório do dataset:", DATA_DIR)

## 3.2 · Obtenção do dataset (opcional e idempotente)

O Task03_Liver tem **~28 GB**. A célula abaixo permite baixá-lo **uma única vez** para o seu
Drive, direto do Colab (tráfego interno do Google, bem mais rápido que baixar no seu computador
e reenviar). Ela foi escrita para ser **segura de reexecutar** (princípio de reprodutibilidade,
§6.6):

- se o dataset **já existe** em `DATA_DIR`, ela **não baixa nada**;
- o download só ocorre se você **explicitamente** definir `BAIXAR_DATASET = True` — assim,
  reexecutar o notebook nunca dispara um download de 28 GB por acidente.

Há duas fontes possíveis (escolha em `FONTE`): o **espelho AWS S3** do MONAI (`wget` direto) ou
o **Google Drive oficial** (via `gdown`, informando o ID do `Task03_Liver.tar`).

> ⚠️ **Antes de baixar:** confirme que há **~30 GB livres** no Drive e que o link/ID está
> vigente (veja *medicaldecathlon.com*). A extração para o Drive pode **demorar bastante** — é
> normal. Faça uma vez; nas próximas sessões, basta montar o Drive.
>
> 💡 O `.tar` oficial já contém a pasta `Task03_Liver/` na **estrutura oficial** — por isso ele
> é extraído no diretório *pai* de `DATA_DIR`, recriando exatamente o layout esperado (§6.1).

In [ ]:
import sys, subprocess

# --- Controles (ajuste conforme sua necessidade) ---
BAIXAR_DATASET = False    # deixe False; mude para True apenas na 1ª vez, se o dataset não estiver no Drive
FONTE = "aws"             # "aws" (espelho S3 do MONAI) ou "gdrive" (Google Drive oficial)
GDRIVE_ID = ""            # se FONTE == "gdrive": ID do arquivo Task03_Liver.tar (ver medicaldecathlon.com)

URL_AWS = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task03_Liver.tar"

# Idempotência: se já houver o conteúdo rotulado, não há o que baixar.
ja_existe = (DATA_DIR / "imagesTr").is_dir() and (DATA_DIR / "labelsTr").is_dir()

if ja_existe:
    print(f"✅ Dataset já presente em '{DATA_DIR}' — download desnecessário.")
elif not BAIXAR_DATASET:
    print("ℹ️  Dataset não encontrado e BAIXAR_DATASET=False.")
    print("    Coloque o Task03_Liver no Drive, OU defina BAIXAR_DATASET=True para baixar agora.")
else:
    tar_local = "/content/Task03_Liver.tar"
    destino = DATA_DIR.parent            # o .tar cria .../Task03_Liver/ ao ser extraído aqui
    destino.mkdir(parents=True, exist_ok=True)

    if FONTE == "aws":
        print("Baixando do espelho AWS S3:", URL_AWS)
        subprocess.run(["wget", "-q", "-O", tar_local, URL_AWS], check=True)
    elif FONTE == "gdrive":
        assert GDRIVE_ID, "Defina GDRIVE_ID (veja o link vigente em medicaldecathlon.com)."
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        print("Baixando do Google Drive (ID):", GDRIVE_ID)
        subprocess.run(["gdown", "--fuzzy", GDRIVE_ID, "-O", tar_local], check=True)
    else:
        raise ValueError("FONTE deve ser 'aws' ou 'gdrive'.")

    print(f"Extraindo para '{destino}' … (pode demorar bastante)")
    subprocess.run(["tar", "-xf", tar_local, "-C", str(destino)], check=True)
    print("✅ Download e extração concluídos.")

## 3.3 · Validação da estrutura e metadados

Antes de tocar em qualquer arquivo, confirmamos que a **estrutura oficial existe**. Em IA
médica, um erro silencioso de caminho pode contaminar todo o experimento — por isso, seguindo
o DPC §7.1, *qualquer inconsistência interrompe a execução* com uma mensagem clara, em vez de
prosseguir com dados errados.

Também lemos o `dataset.json` oficial: ele documenta a modalidade (CT), o significado dos
rótulos (`0` fundo, `1` fígado, `2` lesão) e as contagens esperadas — que usamos para conferir
o que encontramos em disco.

In [ ]:
import json

# Estrutura oficial esperada (DPC §6.1 — apenas verificamos; não modificamos nada).
imagesTr    = DATA_DIR / "imagesTr"
labelsTr    = DATA_DIR / "labelsTr"
imagesTs    = DATA_DIR / "imagesTs"
dataset_json = DATA_DIR / "dataset.json"

# Diretórios rotulados são obrigatórios; sua ausência interrompe a execução (§7.1).
obrigatorios = {"imagesTr/": imagesTr, "labelsTr/": labelsTr}
faltando = [nome for nome, caminho in obrigatorios.items() if not caminho.is_dir()]
if faltando:
    raise FileNotFoundError(
        f"Estrutura oficial do MSD não encontrada em '{DATA_DIR}'. "
        f"Faltando: {', '.join(faltando)}. Verifique o DATA_DIR e a montagem do Drive."
    )
print("✅ Estrutura validada: imagesTr/ e labelsTr/ presentes.")
if not imagesTs.is_dir():
    print("ℹ️  imagesTs/ ausente — sem impacto (não usamos o teste não rotulado do MSD).")

# Metadados oficiais (quando disponíveis).
if dataset_json.is_file():
    with open(dataset_json, encoding="utf-8") as f:
        META = json.load(f)
    print("\nMetadados oficiais (dataset.json):")
    print("   Nome        :", META.get("name"))
    print("   Modalidade  :", META.get("modality"))
    print("   Rótulos     :", META.get("labels"))
    print("   numTraining :", META.get("numTraining"))
    print("   numTest     :", META.get("numTest"))
else:
    META = None
    print("\nℹ️  dataset.json ausente — seguiremos apenas pela varredura de arquivos.")

## 3.4 · Pareamento imagem–máscara

Agora listamos os volumes e as máscaras e verificamos a **correspondência 1:1** entre eles.
Dois cuidados do DPC §7.1:

- **Ordenação estável** com `sorted(glob(...))`: garante que a *n*-ésima imagem case com a
  *n*-ésima máscara, de forma reprodutível;
- **Arquivos ocultos**: descartamos entradas iniciadas por `.` (ex.: `._liver_0.nii.gz`,
  artefatos de sistemas de arquivos) que, se não filtradas, quebrariam o pareamento.

O resultado é a lista `data_dicts` — uma lista de dicionários `{"image": ..., "label": ...}`,
o formato que os *transforms* e *datasets* do MONAI consomem diretamente.

In [ ]:
import os
from glob import glob

def listar_nii(pasta: Path) -> list[str]:
    # sorted(glob(...)) => ordenação estável e reprodutível (DPC §7.1).
    arquivos = sorted(glob(os.path.join(str(pasta), "*.nii.gz")))
    # Ignora arquivos ocultos (ex.: '._liver_0.nii.gz').
    return [a for a in arquivos if not os.path.basename(a).startswith(".")]

vol_paths = listar_nii(imagesTr)
seg_paths = listar_nii(labelsTr)

# Quantidades devem coincidir...
if len(vol_paths) != len(seg_paths):
    raise RuntimeError(
        f"Nº de imagens ({len(vol_paths)}) difere do nº de máscaras ({len(seg_paths)})."
    )

# ...e cada imagem deve ter máscara de mesmo nome.
for v, s in zip(vol_paths, seg_paths):
    if os.path.basename(v) != os.path.basename(s):
        raise RuntimeError(
            f"Imagem e máscara não correspondem: '{os.path.basename(v)}' x '{os.path.basename(s)}'."
        )

data_dicts = [{"image": v, "label": s} for v, s in zip(vol_paths, seg_paths)]
print(f"✅ {len(data_dicts)} pares imagem/máscara rotulados, com correspondência 1:1 verificada.")

## 3.5 · Integridade dos arquivos

Correspondência de nomes não basta: imagem e máscara precisam descrever o **mesmo volume
físico**. Verificamos, para cada par, que as **dimensões espaciais coincidem** (§7.1). Lemos
apenas o *cabeçalho* NIfTI (via `nibabel`, sem carregar os voxels), o que torna a checagem
rápida mesmo sobre dezenas de exames.

In [ ]:
import numpy as np
import nibabel as nib
from tqdm.auto import tqdm

problemas = []
formas = []
for par in tqdm(data_dicts, desc="Verificando integridade"):
    forma_img = nib.load(par["image"]).header.get_data_shape()
    forma_seg = nib.load(par["label"]).header.get_data_shape()
    if forma_img != forma_seg:
        problemas.append((os.path.basename(par["image"]), forma_img, forma_seg))
    formas.append(forma_img)

if problemas:
    for nome, fi, fs in problemas[:10]:
        print(f"  {nome}: imagem {fi} x máscara {fs}")
    raise RuntimeError(f"{len(problemas)} par(es) com dimensões incompatíveis imagem/máscara.")

formas = np.array([f[:3] for f in formas])
print("✅ Integridade OK — imagem e máscara com mesmas dimensões em todos os exames.")
print(f"   Cortes axiais (eixo Z): mín {formas[:, 2].min()}, "
      f"máx {formas[:, 2].max()} — variação anatômica esperada entre pacientes.")

## 3.6 · Divisão por paciente: treino / validação / teste

Chegamos a uma das decisões mais sensíveis do protocolo (§6.1). A divisão segue **separação
estrita por paciente**: exames de um mesmo indivíduo nunca aparecem em conjuntos diferentes —
o que elimina uma das formas mais comuns de **vazamento de dados (*data leakage*)** em
aplicações médicas. No Task03_Liver, cada arquivo corresponde a um paciente distinto, então
uma divisão por arquivo já satisfaz esse critério.

Usamos as proporções **70 / 15 / 15** definidas em `CONFIG` (`val_split`, `test_split`). Para
particionar, empregamos o componente **oficial** `monai.data.partition_dataset` (princípio
§5.4), com `seed=CONFIG.seed` — garantindo que a *mesma divisão* se repita em toda execução
(reprodutibilidade, §6.2). Ao final, uma verificação anti-vazamento confirma que os três
conjuntos são **disjuntos** (§7.1).

- **Treino** — otimiza os parâmetros da rede;
- **Validação** — monitora o treino e orientará a parada antecipada;
- **Teste** — permanece **isolado** até a avaliação final.

In [ ]:
from monai.data import partition_dataset

# Proporções relativas [treino, validação, teste] a partir do protocolo (CONFIG).
train_frac = 1.0 - CONFIG.val_split - CONFIG.test_split
ratios = [train_frac, CONFIG.val_split, CONFIG.test_split]

train_files, val_files, test_files = partition_dataset(
    data_dicts,
    ratios=ratios,
    shuffle=True,          # embaralha antes de dividir...
    seed=CONFIG.seed,      # ...de forma reprodutível.
)

print(f"Divisão por paciente (SEED={CONFIG.seed}) — proporções {ratios}:")
print(f"   Treino    : {len(train_files):>3} exames")
print(f"   Validação : {len(val_files):>3} exames")
print(f"   Teste     : {len(test_files):>3} exames  (isolado até a avaliação final)")

# Verificação anti-vazamento (§7.1): conjuntos disjuntos e cobertura total.
s_tr = {d["image"] for d in train_files}
s_va = {d["image"] for d in val_files}
s_te = {d["image"] for d in test_files}
assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), \
    "VAZAMENTO: há exames repetidos entre conjuntos!"
assert len(s_tr) + len(s_va) + len(s_te) == len(data_dicts), "Perda/duplicação de exames na divisão!"
print("✅ Sem sobreposição entre treino/validação/teste — separação por paciente garantida.")

## 3.7 · Exploração de um exemplo

Números não substituem o olhar. Antes de seguir, carregamos **um** exame (apenas para
inspeção — nenhuma transformação do dataset é aplicada) e olhamos um corte axial central, a
máscara e a sobreposição. Isso confirma, de forma qualitativa, que imagens e rótulos estão
coerentes.

Note os **rótulos presentes**: o MSD marca `1` para o fígado e `2` para lesões hepáticas. Como
o escopo da v1.0 é o **órgão** (§4.3), esses valores serão unificados em um único primeiro plano
(`label > 0`) na Seção 4 — aqui apenas observamos que ambos existem.

> 🧭 A imagem é exibida em sua orientação **crua** (a padronização anatômica RAS acontece no
> pré-processamento). Aplicamos uma janela `[-200, 200] HU` **apenas para exibição**, o que
> realça os tecidos moles do abdome.

In [ ]:
import matplotlib.pyplot as plt
from monai.transforms import LoadImage

# Carregamento CRU de um exemplo (image_only=True devolve apenas o tensor da imagem).
carregar = LoadImage(image_only=True, ensure_channel_first=False)
exemplo = train_files[0]
img = carregar(exemplo["image"]).numpy()
seg = carregar(exemplo["label"]).numpy()

print("Exemplo:", os.path.basename(exemplo["image"]))
print("   Dimensões       :", img.shape)
print("   Intensidade (HU): min", float(img.min()), "| max", float(img.max()))
print("   Rótulos presentes:", [int(v) for v in np.unique(seg)])

z = img.shape[2] // 2                       # corte axial central
img_disp = np.clip(img[:, :, z], -200, 200)  # janela apenas para exibição
seg_disp = seg[:, :, z]

plt.figure(figsize=(13, 5))
plt.subplot(1, 3, 1)
plt.title(f"CT — corte axial z={z}"); plt.imshow(img_disp, cmap="gray"); plt.axis("off")
plt.subplot(1, 3, 2)
plt.title("Máscara (fígado + lesão)"); plt.imshow(seg_disp); plt.axis("off")
plt.subplot(1, 3, 3)
plt.title("Sobreposição"); plt.imshow(img_disp, cmap="gray")
plt.imshow(np.ma.masked_where(seg_disp == 0, seg_disp), alpha=0.5, cmap="autumn"); plt.axis("off")
plt.tight_layout(); plt.show()

## ✅ Resumo da Seção 3

A camada de dados está pronta e auditada:

- localizamos o dataset via Google Drive (`DATA_DIR`) na **estrutura oficial do MSD**, intocada (§6.1);
- **validamos** estrutura, metadados, pareamento 1:1 e integridade dimensional (§7.1);
- dividimos o conjunto rotulado em **treino / validação / teste** por paciente (70/15/15), de
  forma reprodutível e **sem vazamento** — com o teste isolado até o fim (§6.1);
- **exploramos** um exemplo, confirmando a coerência entre imagem e máscara.

As listas `train_files`, `val_files` e `test_files` (dicionários `{"image", "label"}`) são a
entrada da próxima etapa. **Nenhum voxel foi transformado até aqui.**

**➡️ Próxima seção — Pré-processamento.** Definiremos, com *transforms* do MONAI, a cadeia que
prepara cada exame para a rede: leitura, padronização de orientação (RAS), reamostragem
(`pixdim`), normalização de intensidade (`a_min/a_max`), recorte de ROI e redimensionamento
(`spatial_size`) — além do **data augmentation** aplicado *somente* ao treino (§6.3).